### Imports and functions

In [ ]:
from torch.utils.data import DataLoader, random_split
import torch.nn.functional as F
import torch.nn as nn
import torch as tc

import matplotlib.pyplot as plt
from dataset_creator import *
from datetime import datetime
import pandas as pd
import numpy as np
from functions import *

from sklearn.preprocessing import StandardScaler

device = tc.device("cuda" if tc.cuda.is_available() else "cpu")

import warnings
warnings.filterwarnings("ignore")
plt.style.use('seaborn-v0_8')


### Examples

In [ ]:
model = FusionUNetFNO()
model.load_state_dict(tc.load("model_PS.pth"))
model.eval()

In [ ]:
length = 2000
file = "./data_tests/test_inputs.csv"
X = pd.read_csv(file).values
X = X.reshape(-1, 3, 6000)

file = "./data_tests/test_outputs.csv"
y = pd.read_csv(file).values
y_true = tc.tensor(y, dtype=tc.float32)

X_tensor = tc.tensor(X, dtype=tc.float32)
X_tensor, y_true = mix_batch_events_torch(X_tensor, y_true,
                    noise_prob=0.0,
                    mix_prob =0.0,
                    num_events_range=(1,1),
                    device='cpu')

b, c, _ = X_tensor.shape
datos_normalizados = np.zeros_like(X_tensor)
scaler = StandardScaler()

for i in range(b):
    for j in range(c):
        señal = X_tensor[i, j, :].reshape(-1, 1).numpy()
        señal_norm = scaler.fit_transform(señal).flatten()
        datos_normalizados[i, j, :] = señal_norm

X_tensor = tc.tensor( datos_normalizados, dtype=tc.float32 )
X_tensor = X_tensor.reshape(b, c, 3, length)
X_tensor = X_tensor.permute(0, 2, 1, 3)
X_tensor = X_tensor.reshape(b * 3, c, length)

y_true = y_true.reshape(b, 2, 3, length)
y_true = y_true.permute(0, 2, 1, 3)
y_true = y_true.reshape(b * 3, 2, length)
y_true = y_true.numpy()

with tc.no_grad():
    predLabels = model( X_tensor )
predLabels = tc.clip( predLabels, min=0, max=1 )
